In [1]:
!pip install -U outlines langchain-classic langchain-community langchain-chroma langchain-core langchain-text-splitters requests rank_bm25 langchain_huggingface

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.4/108.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB

In [2]:
import pandas as pd
import os
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import GenerationConfig
from transformers import  BitsAndBytesConfig

from pydantic import BaseModel,Field
import outlines
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
import torch
import warnings
warnings.filterwarnings("ignore")
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFacePipeline

from langchain_core.prompts import PromptTemplate
from transformers import pipeline, set_seed

/tmp/ipykernel_2365/3322927846.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [3]:
def load_vector_store(model_name):

  embeddings = HuggingFaceEmbeddings(model_name=model_name)
  loaded_vector_store = Chroma(
    persist_directory="/content/drive/MyDrive/Colab Notebooks/RAG/vector_store/Chroma",
    embedding_function=embeddings,
    collection_name="ai_knowledge_base"

  )

  return loaded_vector_store

#### Pass same model as passed in creation of vector store
model_name = 'all-MiniLM-L6-v2'
vector_store = load_vector_store(model_name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### vector store retriver we can retrieve top k tontext using below method

* https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore/as_retriever
* Only retrieve documents that have a relevance score Above a certain threshold
docsearch.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.8},
)

* query = 'What was the amount of children murdered?'
* results = retriever.invoke(query)

* normalized_results = vector_store.similarity_search_with_relevance_scores(query, k=2)

### Enable Thinking Model in Structure Output

* Because you are using a reasoning model (enable_thinking=True), forcing structured output at the logit level can break the raw thinking blocks (<think>...</think>) if you aren't careful. To get both raw reasoning and a guaranteed structured final answer, you should inject the JSON schema requirements via a system prompt and parse the final text object.

In [4]:
#### Load Model and Tokenizer
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
vanila_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

generator = pipeline('text-generation', model="facebook/opt-125m", do_sample=True)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Creating Different type of retriver to get better Context

##### Hybrid Retrieval

* Combines dense vector search and keyword-based search(BM 25) using LangChain EnsembleRetriever get result from running them simultaneously against a single user query, and merging their outputs into a single, reranked list of documents using the Reciprocal Rank Fusion (RRF) algorithm

##### Multi-Query Retrieval

* Multi-Query Retrieval in LangChain is an advanced query transformation technique that uses an LLM to generate multiple variations of a user's initial question from different perspectives. By executing these variations simultaneously against a vector database and returning a unique union of the retrieved documents, it overcomes the distance-based limitations of standard semantic search solves problem like Query Rewriting,LLM Variation,Parallel Execution, De-duplication

In [5]:
#### Normal retriver with similarity search
def context_creation(query,top_k):

  raw_results = vector_store.similarity_search_with_score(query, k=top_k)
  context = '\n'.join([doc[0].page_content for doc in raw_results])
  return context

##

In [6]:
### Hybrid Retriver

base_data_path = '/content/drive/MyDrive/Colab Notebooks/RAG/Data'
train_dataframe = pd.read_excel(os.path.join(base_data_path,'train.xlsx'))
val_dataframe = pd.read_excel(os.path.join(base_data_path,'val.xlsx'))
test_dataframe = pd.read_excel(os.path.join(base_data_path,'test.xlsx'))

def create_total_context():

  total_data = []
  for ind in range(0,len(train_dataframe)):
    temp_list = [i for i in train_dataframe['context'][ind].split("\n") if i]
    total_data.extend(temp_list)
  for ind in range(0,len(val_dataframe)):
    temp_list = [i for i in val_dataframe['context'][ind].split("\n") if i]
    total_data.extend(temp_list)
  for ind in range(0,len(test_dataframe)):
    temp_list = [i for i in test_dataframe['context'][ind].split("\n") if i]
    total_data.extend(temp_list)

  return list(set(total_data))


total_context_list = create_total_context()
### Creating langchain Document Object

Documents = []
for i,value in enumerate(total_context_list):
  Documents.append(Document(page_content=value, metadata={"chunk_ind": i+1}))

bm25_retriever = BM25Retriever.from_documents(Documents)
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

vector_store_hybrid = Chroma.from_documents(
      documents=Documents,
      embedding=embeddings,
  )

def hybrid_retrival(query,top_k):



  bm25_retriever.k = top_k
  vector_retriever = vector_store_hybrid.as_retriever(search_kwargs={"k": top_k})
  ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]  # Balances the importance of BM25 vs Vector search
  )

  results = ensemble_retriever.invoke(query)
  context = '\n'.join([doc.page_content for doc in results])

  return context


#### Multi-Query Retrieval
def multi_query_retrival(query,top_k):

  custom_template = """You are an AI language model assistant. Your task is to generate 3 different versions of the given user question to retrieve relevant documents from a vector database. By generating multiple perspectives on the user question, your goal is to helpthe user overcome some of the limitations of distance-based similarity search.
  Original question: {question}"""
  CUSTOM_PROMPT = PromptTemplate(input_variables=["question"],template=custom_template)

  set_seed(32)
  llm = HuggingFacePipeline(pipeline=generator)
  base_retriever = vector_store.as_retriever(search_kwargs={"k": top_k})
  mq_retriever = MultiQueryRetriever.from_llm(
      retriever=base_retriever,
      llm=llm,
      prompt=CUSTOM_PROMPT,
      include_original=True
  )
  docs = mq_retriever.invoke(query)
  context = '\n'.join([doc.page_content for doc in docs])

  return context


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
def get_promt():

  system_prompt = '''You are a Question Answering Assistant.

  Your task is to answer the user's question using ONLY the provided context.

  Rules:
  1. Use only information explicitly present in the context.
  2. Do not use external knowledge.
  3. Do not infer or assume missing information.
  4. If the answer is not present in the context, return:
    {
      "answer": null,
      "confidence": 0
    }
  5. Confidence must be an integer between 0 and 100.
  6. Return only valid JSON.
  7. Do not include explanations, reasoning, citations, or additional fields.

  Output Format:

  {"answer": "<answer>","confidence": <0-100>}
  '''

  user_prompt = '''Context:
  {context}

  Question:
  {question}

  Answer the question using only the provided context.
  '''
  return system_prompt ,user_prompt


In [8]:
class RAGCLass(BaseModel):
    Answer: str  = Field(description="Answer to the Question")
    Confidence_Score: float = Field(description="confidence score of Anser")

In [9]:
#### Refering to the Hugging Face Documentation

def generate_text_function(model_type,query,top_k,enable_thinking=False):

  ### Different Retieal technique
  # context = context_creation(query,top_k)
  # context = hybrid_retrival(query,top_k*2)
  context = multi_query_retrival(query,top_k)


  system_prompt ,user_prompt = get_promt()

  user_prompt = user_prompt.format(context = context , question = query)

  messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt}
  ]

  generation_config = GenerationConfig(
    temperature=0.1,
    top_p=0.9,
    top_k=1,
    max_new_tokens=32768,
    seed = 42,
    do_sample=True # Crucial for Qwen models to respect temperature
  )

  text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=enable_thinking # Switches between thinking and non-thinking modes. Default is True.
  )
  model_inputs = tokenizer([text], return_tensors="pt").to(model_type.device)

  # conduct text completion
  generated_ids = model_type.generate(
      **model_inputs,
      generation_config=generation_config,
  )
  output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

  # parsing thinking content
  try:
      # rindex finding 151668 (</think>)
      index = len(output_ids) - output_ids[::-1].index(151668)
  except ValueError:
      index = 0

  content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
  if enable_thinking:
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
  else:
    thinking_content = None


  ### Structure Output
  outlines_model = outlines.from_transformers(
    model_type,
    tokenizer
  )
  generator = outlines.Generator(
    outlines_model,
    RAGCLass

  )
  structured_prompt = f"""
  Convert the following answer into the schema.

  Answer:
  {content}
  """
  result = generator(structured_prompt,max_new_tokens=32768)

  structured_response = RAGCLass.model_validate_json(result)

  return structured_response,thinking_content




In [10]:
query = 'When was Pandher sentenced to death?'
top_k = 20

normal_output,thinking = generate_text_function(vanila_model,query,top_k,True)
print(normal_output)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Answer='February 2009' Confidence_Score=100.0


In [11]:
print(thinking)

<think>
Okay, let's tackle this question. The user is asking when Pandher was sentenced to death. I need to look through the provided context to find the answer.

First, I'll scan through the text. The context is a mix of different topics, including a teaching session, a story about a murder, and some news about events in the US. 

Looking for mentions of Pandher, I notice that in the section about the U.S. Supreme Court case, there's a line: "Moninder Singh Pandher was sentenced to death by a lower court in February." Then later, there's another mention: "Pandher was not named a main suspect by investigators initially, but was summoned as co-accused during the trial, Kochar said." 

Wait, but the question is about when he was sentenced. The sentence was given by a lower court in February. So the answer should be February. Let me double-check to make sure there's no conflicting information. The context says "Moninder Singh Pandher was sentenced to death by a lower court in February." N

In [12]:
### temp code
base_data_path = '/content/drive/MyDrive/Colab Notebooks/RAG/Data'
train_dataframe = pd.read_excel(os.path.join(base_data_path,'train.xlsx'))
# context = '\n'.join(train_dataframe['context'].tolist()[:10])


In [13]:
train_dataframe.head()

,Unnamed: 0,context,question,answers
0,0,"NEW DELHI, India (CNN) -- A high court in nort...",What was the amount of children murdered?,['19']
1,1,"NEW DELHI, India (CNN) -- A high court in nort...",When was Pandher sentenced to death?,['February.']
2,2,"NEW DELHI, India (CNN) -- A high court in nort...",The court aquitted Moninder Singh Pandher of w...,['rape and murder']
3,3,"NEW DELHI, India (CNN) -- A high court in nort...",who was acquitted,['Moninder Singh Pandher']
4,4,"NEW DELHI, India (CNN) -- A high court in nort...",who was sentenced,['Moninder Singh Pandher']


In [14]:
# ind = 2
# print(train_dataframe['question'][ind])
# print(train_dataframe['answers'][ind])
# print(train_dataframe['context'][ind])

In [15]:
# print(context)

In [30]:
raw_results = vector_store.similarity_search_with_score(query, k=top_k)
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [69]:
from torch.nn import Sigmoid
def rerank_documents(query, base_ret_results, top_k=3):

    pairs = [(query, item[0].page_content) for item in base_ret_results]
    scores = reranker.predict(pairs,activation_fn=Sigmoid())


    ranked = sorted(
        zip(scores, base_ret_results),
        reverse=True,
        key=lambda x: x[0]
    )

    return [item for _, item in ranked[:top_k]]

In [70]:
out = rerank_documents(query , raw_results)

[np.float32(0.9997), np.float32(0.0313), np.float32(0.0199), np.float32(0.0037), np.float32(0.0), np.float32(0.0039), np.float32(0.0819), np.float32(0.0064), np.float32(0.0), np.float32(0.0024), np.float32(0.0006), np.float32(0.0036), np.float32(0.0), np.float32(0.6899), np.float32(0.0004), np.float32(0.0014), np.float32(1e-04), np.float32(0.0133), np.float32(0.7147), np.float32(0.0013)]
